import os
for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

"""
U-Net and Attention U-Net for Semantic Segmentation
=====================================================

A clean, linear (top-to-bottom) rewrite of the two U-Net models defined
in `Experimentation.ipynb` (`UNet` and `UNetAM`), using the Keras
Functional API.

WHAT CHANGED VS. THE ORIGINAL NOTEBOOK (behavior-preserving cleanup):
  - Wildcard imports (`from keras.layers import *`) replaced with
    explicit imports, so it's clear where every function comes from.
  - `keras.optimizers.adam_v2.Adam` (deprecated path) replaced with
    `tf.keras.optimizers.Adam`.
  - The two model-building cells no longer depend on each other's
    leftover variables/functions -- both `convBlock` and `convBlock2`
    are defined up front, in one place.
  - Both models kept their exact layer-by-layer logic, including the
    two issues noted below.
"""


In [ ]:
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, Conv2DTranspose, Activation, MaxPooling2D,
    UpSampling2D, concatenate, add, Lambda, Dropout,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall, BinaryIoU
 
 
# --------------------------------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------------------------------
INPUT_SIZE = (224, 224, 3)   # (height, width, channels) -- 224x224 color (RGB) images
LEARNING_RATE = 1e-4
DROP_RATE = 0.25             # applied via Dropout layers at each model's bottleneck (see build_unet/build_attention_unet) -- unused in the original notebook, where the dropout lines were commented out
FILTER_BASE = 16             # only used by UNetAM; UNet's filter counts are hard-coded (64/128/256/512/1024)
 
# Width (filters in the first block) of the U-Nets in the attention-gate
# placement comparison (the "abl_*" models in MODEL_REGISTRY below). The
# standard U-Net is 64 wide ("unet"); 32 needs roughly a quarter of the
# compute and memory, which matters when several variants have to be
# trained. Every "abl_*" model must share this value for the comparison to be
# fair, and changing it changes what saved "abl_*" weights fit.
ABLATION_BASE_FILTERS = 32
 
# The test set measured at ~10% forest / 90% non-forest pixels (a real, fairly
# imbalanced split). POS_WEIGHT makes mistakes on forest pixels count roughly
# this many times more than mistakes on non-forest pixels during training, so
# the model can't just get a high score by predicting "non-forest" everywhere.
# Rule of thumb: non_forest_fraction / forest_fraction -- recompute this from
# your actual TRAINING set's forest fraction if it differs noticeably from 10%.
POS_WEIGHT = 9.0
 
 
# --------------------------------------------------------------------------
# 2. LOSS FUNCTION (weighted for class imbalance) + TRAINING METRICS
# --------------------------------------------------------------------------
def weighted_binary_crossentropy(pos_weight=POS_WEIGHT):
    """Like standard binary_crossentropy, but mistakes on forest pixels (the
    minority class) are penalized `pos_weight` times more than mistakes on
    non-forest pixels. Without this, a model can reach ~90% pixel accuracy
    on this dataset just by predicting "non-forest" everywhere -- this loss
    makes that a poor strategy instead of a free win.
    """
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)   # avoid log(0)
        per_pixel_loss = -(
            pos_weight * y_true * tf.math.log(y_pred)
            + (1 - y_true) * tf.math.log(1 - y_pred)
        )
        return tf.reduce_mean(per_pixel_loss)
    return loss
 
 
def dice_loss(y_true, y_pred, smooth=1.0):
    """1 - Dice coefficient. Measures region OVERLAP directly (how much the
    predicted forest area and true forest area agree), rather than scoring
    each pixel independently like cross-entropy does. This gives a much
    stronger gradient signal toward actually predicting SOME forest pixels,
    which is specifically useful when a model has collapsed to predicting
    the majority class (solid black / all non-forest) everywhere -- a
    failure plain weighted cross-entropy can still fall into on its own.
    """
    y_true_flat = tf.reshape(y_true, [-1])
    y_pred_flat = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_flat * y_pred_flat)
    dice_coefficient = (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true_flat) + tf.reduce_sum(y_pred_flat) + smooth
    )
    return 1.0 - dice_coefficient
 
 
def combined_loss(pos_weight=POS_WEIGHT):
    """weighted_binary_crossentropy + dice_loss. The two push the model in
    complementary ways -- weighted BCE penalizes individual wrong pixels,
    Dice rewards getting the overall forest-shaped REGION right -- and
    together they're considerably more resistant to the model collapsing
    to "always predict non-forest" than either one is on its own.
    """
    wbce = weighted_binary_crossentropy(pos_weight)
 
    def loss(y_true, y_pred):
        return wbce(y_true, y_pred) + dice_loss(y_true, y_pred)
    return loss
 
 
def segmentation_metrics(threshold=0.5):
    """Metrics Keras reports for every epoch, on both the training data and
    the validation data. "Forest" (1) is the positive class throughout, the
    same convention predict.py uses.
 
    Each value is pooled over every pixel seen in that epoch (true/false
    positives are accumulated across all batches, then divided once at the
    end), so the numbers line up with the "Overall (pooled)" rows predict.py
    prints, not with its per-image averages. F1 is not listed here because
    train.py derives it from precision and recall, which gives exactly the
    same pooled value.
 
    NOTE: every metric gets an explicit name. Without one, Keras renames
    repeated metrics ("precision_1", "precision_2", ...) each time a model is
    built in the same session, which would break the history keys that
    train.py looks up.
    """
    return [
        BinaryAccuracy(name="accuracy", threshold=threshold),
        Precision(name="precision", thresholds=threshold),
        Recall(name="recall", thresholds=threshold),
        BinaryIoU(target_class_ids=[1], threshold=threshold, name="iou"),   # IoU of the forest class only
    ]
 
 
# --------------------------------------------------------------------------
# 3. SHARED BUILDING BLOCKS
# --------------------------------------------------------------------------
def convBlock(input_tensor, filters, kernel, kernel_init="he_normal", act="relu", transpose=False):
    """Single Conv (or Conv2DTranspose) + activation. Used by UNet's contraction/
    expansion stages and by UNetAM's final 1x1 sigmoid output layer."""
    if not transpose:
        conv = Conv2D(filters, kernel, padding="same", kernel_initializer=kernel_init)(input_tensor)
    else:
        conv = Conv2DTranspose(filters, kernel, padding="same", kernel_initializer=kernel_init)(input_tensor)
    conv = Activation(act)(conv)
    return conv
 
 
def convBlock2(input_tensor, filters, kernel, kernel_init="he_normal", act="relu", transpose=False):
    """Two Conv (or Conv2DTranspose) + activation layers back to back. Used by UNetAM."""
    if not transpose:
        conv = Conv2D(filters, kernel, padding="same", kernel_initializer=kernel_init)(input_tensor)
        conv = Activation(act)(conv)
        conv = Conv2D(filters, kernel, padding="same", kernel_initializer=kernel_init)(conv)
        conv = Activation(act)(conv)
    else:
        conv = Conv2DTranspose(filters, kernel, padding="same", kernel_initializer=kernel_init)(input_tensor)
        conv = Activation(act)(conv)
        conv = Conv2DTranspose(filters, kernel, padding="same", kernel_initializer=kernel_init)(conv)
        conv = Activation(act)(conv)
    return conv
 
 
def attention_block(x, gating, inter_shape, drop_rate=0.25):
    """Attention gate used by UNetAM. Pools `x` down to match the gating
    signal's resolution, combines the two, produces a sigmoid attention
    map, upsamples it back to `x`'s resolution, and multiplies it into `x`."""
    shape_x = K.int_shape(x)
 
    # Process the skip-connection tensor (x) and the gating signal
    theta_x = Conv2D(inter_shape, kernel_size=1, strides=1, padding="same",
                      kernel_initializer="he_normal", activation=None)(x)
    theta_x = MaxPooling2D((2, 2))(theta_x)
 
    phi_g = Conv2D(inter_shape, kernel_size=1, strides=1, padding="same",
                    kernel_initializer="he_normal", activation=None)(gating)
 
    # Combine, activate, reduce to a single-channel attention map
    concat_xg = add([phi_g, theta_x])
    act_xg = Activation("relu")(concat_xg)
    psi = Conv2D(1, kernel_size=1, strides=1, padding="same",
                 kernel_initializer="he_normal", activation=None)(act_xg)
    sigmoid_xg = Activation("sigmoid")(psi)
    shape_sigmoid = K.int_shape(sigmoid_xg)
 
    # Upsample the attention map back to x's resolution and apply it
    upsample_psi = UpSampling2D(
        interpolation="bilinear",
        size=(shape_x[1] // shape_sigmoid[1], shape_x[2] // shape_sigmoid[2]),
    )(sigmoid_xg)
    # NOTE: Keras 3 (used by current TF versions, e.g. on Kaggle) doesn't allow
    # calling a raw tf.* function directly on a KerasTensor -- it must be wrapped
    # in a Keras layer, which Lambda does for us. The target shape is computed
    # dynamically via tf.shape(x) INSIDE the Lambda (rather than a static Python
    # tuple captured beforehand), because x's batch dimension is flexible
    # (None) now that Input() no longer hard-codes batch_size=1 -- a static
    # shape containing None isn't valid for tf.broadcast_to.
    def _broadcast_and_multiply(tensors):
        upsample_psi, x = tensors
        upsample_psi = tf.broadcast_to(upsample_psi, tf.shape(x))
        return upsample_psi * x
 
    y = Lambda(_broadcast_and_multiply)([upsample_psi, x])
    return y
 
 
# --------------------------------------------------------------------------
# 4. PLAIN U-NET
# --------------------------------------------------------------------------
def build_unet(input_size=INPUT_SIZE, lr=LEARNING_RATE, drop_rate=DROP_RATE,
               attention_levels=(), base_filters=64, name="unet"):
    """U-Net, optionally with spatial attention gates on its skip connections.
 
    attention_levels: which skip connections get an attention gate before
        being concatenated into the decoder. Levels are numbered from the top
        of the "U": level 1 = the highest-resolution skip (same size as the
        input image), level 4 = the deepest skip (1/8 of the input size).
        () = the plain U-Net with no gates; (1, 2, 3, 4) = a gate on every skip.
    base_filters: filters in the first block, doubling at each level. 64 gives
        the standard 64/128/256/512/1024 U-Net.
    """
    unknown = set(attention_levels) - {1, 2, 3, 4}
    if unknown:
        raise ValueError(f"attention_levels may only contain 1, 2, 3 or 4 -- got {sorted(unknown)}")
 
    f = base_filters
    inputs = Input(input_size)   # batch dimension left flexible (was hard-coded to 1)
 
    # Contraction phase
    conv1 = convBlock(inputs, f, 3)
    conv1 = convBlock(conv1, f, 3)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)
 
    conv2 = convBlock(pool1, 2 * f, 3)
    conv2 = convBlock(conv2, 2 * f, 3)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)
 
    conv3 = convBlock(pool2, 4 * f, 3)
    conv3 = convBlock(conv3, 4 * f, 3)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)
 
    conv4 = convBlock(pool3, 8 * f, 3)
    conv4 = convBlock(conv4, 8 * f, 3)
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)
 
    conv5 = convBlock(pool4, 16 * f, 3)
    conv5 = convBlock(conv5, 16 * f, 3)
    conv5 = Dropout(drop_rate)(conv5)   # regularization at the bottleneck -- deepest, most overfit-prone layer
 
    # Expansion phase. Each stage upsamples the previous output and concatenates
    # the matching skip connection -- passed through a spatial attention gate
    # first when that level is in attention_levels. The gate is steered by the
    # previous decoder output (its "gating signal", always at half the skip's
    # resolution), the same way the attention U-Net below does it.
    up6 = Conv2DTranspose(8 * f, kernel_size=2, strides=2, kernel_initializer="he_normal")(conv5)
    skip4 = attention_block(conv4, conv5, 8 * f, drop_rate) if 4 in attention_levels else conv4
    conv6 = convBlock(concatenate([skip4, up6]), 8 * f, 3)
    conv6 = convBlock(conv6, 8 * f, 3)
 
    up7 = Conv2DTranspose(4 * f, kernel_size=2, strides=2, kernel_initializer="he_normal")(conv6)
    skip3 = attention_block(conv3, conv6, 4 * f, drop_rate) if 3 in attention_levels else conv3
    conv7 = convBlock(concatenate([skip3, up7]), 4 * f, 3)
    conv7 = convBlock(conv7, 4 * f, 3)
 
    up8 = Conv2DTranspose(2 * f, kernel_size=2, strides=2, kernel_initializer="he_normal")(conv7)
    skip2 = attention_block(conv2, conv7, 2 * f, drop_rate) if 2 in attention_levels else conv2
    conv8 = convBlock(concatenate([skip2, up8]), 2 * f, 3)
    conv8 = convBlock(conv8, 2 * f, 3)
 
    up9 = Conv2DTranspose(f, kernel_size=2, strides=2, kernel_initializer="he_normal")(conv8)
    skip1 = attention_block(conv1, conv8, f, drop_rate) if 1 in attention_levels else conv1
    conv9 = convBlock(concatenate([skip1, up9]), f, 3)
    conv9 = convBlock(conv9, f, 3)
 
    # Output layer
    conv10 = convBlock(conv9, 1, 1, act="sigmoid")
 
    model = Model(inputs, conv10, name=name)
    model.compile(optimizer=Adam(learning_rate=lr), loss=combined_loss(), metrics=segmentation_metrics())
    return model
 
 
# --------------------------------------------------------------------------
# 5. ATTENTION U-NET
# --------------------------------------------------------------------------
def build_attention_unet(input_size=INPUT_SIZE, drop_rate=DROP_RATE, lr=LEARNING_RATE, filter_base=FILTER_BASE):
    inputs = Input(input_size)   # batch dimension left flexible (was hard-coded to 1)
 
    # Contraction phase
    conv = convBlock2(inputs, filter_base, 3)
 
    conv0 = MaxPooling2D(pool_size=(2, 2))(conv)
    conv0 = convBlock2(conv0, 2 * filter_base, 3)
 
    pool0 = MaxPooling2D(pool_size=(2, 2))(conv0)
    conv1 = convBlock2(pool0, 4 * filter_base, 3)
 
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)
    conv2 = convBlock2(pool1, 8 * filter_base, 3)
 
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)
    conv3 = convBlock2(pool2, 16 * filter_base, 3)
    conv3 = Dropout(drop_rate)(conv3)   # regularization at the bottleneck -- deepest, most overfit-prone layer
 
    # Expansion phase -- each stage attention-gates its matching skip connection
    up4 = Conv2DTranspose(8 * filter_base, kernel_size=2, strides=2, kernel_initializer="he_normal")(conv3)
    merge4 = attention_block(conv2, conv3, 8 * filter_base, drop_rate)
    conv4 = concatenate([up4, merge4])
    conv4 = convBlock2(conv4, 8 * filter_base, 3)
 
    up5 = Conv2DTranspose(4 * filter_base, kernel_size=2, strides=2, kernel_initializer="he_normal")(conv4)
    merge5 = attention_block(conv1, conv4, 4 * filter_base, drop_rate)
    conv5 = concatenate([up5, merge5])
    conv5 = convBlock2(conv5, 4 * filter_base, 3)
 
    up6 = Conv2DTranspose(2 * filter_base, kernel_size=2, strides=2, kernel_initializer="he_normal")(conv5)
    merge6 = attention_block(conv0, conv5, 2 * filter_base, drop_rate)
    conv6 = concatenate([up6, merge6])
    conv6 = convBlock2(conv6, 2 * filter_base, 3)
 
    up7 = Conv2DTranspose(1 * filter_base, kernel_size=2, strides=2, kernel_initializer="he_normal")(conv6)
    # NOTE: original notebook computes this attention-gated merge (merge7) and concatenates
    # it into conv7, then immediately overwrites conv7 with a plain (non-attention)
    # concatenation on the next line -- so merge7's result is discarded. Kept as-is to
    # match the original behavior; swap the commented line in to actually use attention here.
    merge7 = attention_block(conv, conv6, 1 * filter_base, drop_rate)
    conv7 = concatenate([up7, merge7])
    conv7 = concatenate([up7, conv])   # <-- overwrites the line above (original notebook behavior)
    # conv7 = concatenate([up7, merge7])   # <-- uncomment (and delete the line above) to fix
    conv7 = convBlock2(conv7, 1 * filter_base, 3)
 
    # Output layer
    out = convBlock(conv7, 1, 1, act="sigmoid")
 
    model = Model(inputs, out, name="attention_unet")
    model.compile(optimizer=Adam(learning_rate=lr), loss=combined_loss(), metrics=segmentation_metrics())
    return model
 
 
# --------------------------------------------------------------------------
# 6. MODEL REGISTRY
# --------------------------------------------------------------------------
# Every model that train.py / predict.py / analyze.py can use, by name. Each
# entry is a zero-argument function that builds and compiles a fresh model.
#
#   "unet"            standard 64-wide U-Net, no gates
#   "attention_unet"  the notebook's attention U-Net (its own, narrower design)
#
# Attention-gate PLACEMENT comparison -- the same U-Net (width
# ABLATION_BASE_FILTERS) every time, differing ONLY in which skip connections
# have a spatial attention gate. Level 1 is the highest-resolution skip,
# level 4 the deepest:
#
#   "abl_none"        no gates (the baseline to beat)
#   "abl_L1" ... "abl_L4"   a gate on that one level only
#   "abl_all"         gates on all four levels
#
# To test another combination, add one line, e.g. gates on the two deepest
# levels:  "abl_L3_L4": _unet_variant((3, 4), "abl_L3_L4"),
 
def _unet_variant(levels, name):
    def build():
        return build_unet(attention_levels=levels, base_filters=ABLATION_BASE_FILTERS, name=name)
    return build
 
 
MODEL_REGISTRY = {
    "unet": build_unet,
    "attention_unet": build_attention_unet,
    "abl_none": _unet_variant((), "abl_none"),
    "abl_L1": _unet_variant((1,), "abl_L1"),
    "abl_L2": _unet_variant((2,), "abl_L2"),
    "abl_L3": _unet_variant((3,), "abl_L3"),
    "abl_L4": _unet_variant((4,), "abl_L4"),
    "abl_all": _unet_variant((1, 2, 3, 4), "abl_all"),
}
 
 
# --------------------------------------------------------------------------
# 7. RUN
# --------------------------------------------------------------------------
if __name__ == "__main__":
    unet = build_unet()
    unet.summary()
 
    attention_unet = build_attention_unet()
    attention_unet.summary()
 
    # Typical training call, once your data is loaded:
    #
    #   history = attention_unet.fit(
    #       train_images, train_masks,
    #       validation_data=(val_images, val_masks),
    #       batch_size=4,   # any batch size now works -- Input is no longer locked to 1
    #       epochs=50,
    #   )

"""
Data Loader — Forest / Non-Forest Segmentation (grayscale PNG/JPG)
====================================================================
 
Loads paired image/mask files from the `data/` folder structure laid out
in the setup guide, resizes them to a consistent size, and returns them
as NumPy arrays ready to feed into `build_attention_unet()` /
`build_unet()` from `unet_models.py`.
"""
 

In [ ]:
import os
import numpy as np
from PIL import Image
 
 
# --------------------------------------------------------------------------
# 1. CONFIGURATION — change these to match your actual data
# --------------------------------------------------------------------------
IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
 
# File extensions treated as images / masks (lowercase, with the dot). Matching
# is case-insensitive, so .BMP files are picked up too. Add more here if you
# have a mix of formats, e.g. (".bmp", ".png").
IMAGE_EXTENSIONS = (".bmp",)
MASK_EXTENSIONS = (".bmp",)
 
# Tiles that sit at the edge of the original source image get padded with
# solid black to fill out to the full 224x224 slice size. These two settings
# control how such tiles get detected and excluded from the dataset entirely.
BLACK_PIXEL_VALUE_THRESHOLD = 0.02   # a pixel counts as "black" if every channel is below this (near 0)
MAX_BLACK_FRACTION = 0.15            # skip a tile if more than this fraction of it is black padding
 
 
 
# --------------------------------------------------------------------------
# 2. FILE DISCOVERY + SINGLE IMAGE / MASK LOADERS
# --------------------------------------------------------------------------
def _index_by_stem(directory, extensions):
    """Maps filename-without-extension -> full filename, for every file in
    `directory` whose extension is in `extensions` (case-insensitive).
 
    Also returns the names of files that were ignored for having some other
    extension, so a wrong-format dataset gets reported instead of silently
    loading zero images.
    """
    by_stem, ignored = {}, []
    for filename in sorted(os.listdir(directory)):
        stem, ext = os.path.splitext(filename)
        if ext.lower() in extensions:
            by_stem[stem] = filename
        else:
            ignored.append(filename)
    return by_stem, ignored
 
 
def load_image(path, target_size=(IMAGE_WIDTH, IMAGE_HEIGHT)):
    """Load a color image, resize it, normalize pixel values to [0, 1]."""
    img = Image.open(path).convert("RGB")        # "RGB" = 3-channel color
    img = img.resize(target_size, Image.BILINEAR)
    arr = np.array(img, dtype="float32") / 255.0
    arr = arr.reshape(target_size[1], target_size[0], 3)   # (H, W, 3)
    return arr
 
 
def is_mostly_valid(image_arr, max_black_fraction=MAX_BLACK_FRACTION,
                     black_value_threshold=BLACK_PIXEL_VALUE_THRESHOLD):
    """Returns False if too much of an already-loaded image is solid black --
    a sign this tile sat at the edge of the original source image and got
    padded to fill out the slice size, rather than containing real data
    across its full area. `image_arr` should already be normalized to [0, 1]
    (i.e. straight from load_image())."""
    is_black_pixel = np.all(image_arr < black_value_threshold, axis=-1)   # per-pixel, across all channels
    black_fraction = is_black_pixel.mean()
    return black_fraction <= max_black_fraction
 
 
def load_mask(path, target_size=(IMAGE_WIDTH, IMAGE_HEIGHT)):
    """Load a mask, resize it, and binarize it into {0, 1} (forest / non-forest).
 
    NOTE: masks are normalized to their OWN max value before thresholding,
    rather than compared against a fixed MASK_THRESHOLD on a 0-255 scale.
    This makes it work correctly whether the source files encode forest
    pixels as 255 (typical "visual" grayscale masks) OR as 1 (typical
    "label map" masks) -- a fixed threshold of 127 would silently zero out
    an entire 0/1-encoded mask, since 1 is nowhere close to 127.
    """
    mask = Image.open(path)
    if mask.mode == "P":
        # 8-bit palettized image (common for BMP masks exported from paint/GIS
        # tools). convert("L") would translate the palette's COLORS into gray
        # levels, which can scramble the class values -- so read the raw
        # palette INDICES (the actual class labels) instead.
        mask = Image.fromarray(np.array(mask, dtype="uint8"))
    else:
        mask = mask.convert("L")
    mask = mask.resize(target_size, Image.NEAREST)   # NEAREST avoids inventing new class values
    arr = np.array(mask, dtype="float32")
    if arr.max() > 0:
        arr = arr / arr.max()          # scale whatever the encoding is down to a 0-1 range
    arr = (arr > 0.5).astype("float32")   # binarize: 1 = forest, 0 = non-forest
    arr = arr.reshape(target_size[1], target_size[0], 1)
    return arr
 
 
# --------------------------------------------------------------------------
# 3. FOLDER-LEVEL LOADER
# --------------------------------------------------------------------------
def load_dataset(dataset_root, split, target_size=(IMAGE_WIDTH, IMAGE_HEIGHT)):
    """Load every image/mask pair for one split (e.g. split="train").
 
    Expects this layout, matching the Hutan Royal Belum dataset structure:
        dataset_root/images/<split>/...
        dataset_root/masks/<split>/...
    e.g. load_dataset("/kaggle/input/hutan-royal-belum-dataset/Dataset", "train")
 
    Tiles that are mostly black padding (edge-of-source-image tiles -- see
    MAX_BLACK_FRACTION above) are excluded automatically, same as tiles
    missing a matching mask.
 
    Returns two NumPy arrays: images shaped (N, height, width, 3),
    masks shaped (N, height, width, 1).
    """
    images_dir = os.path.join(dataset_root, "images", split)
    masks_dir = os.path.join(dataset_root, "masks", split)
 
    image_files, ignored_images = _index_by_stem(images_dir, IMAGE_EXTENSIONS)
    mask_files, _ = _index_by_stem(masks_dir, MASK_EXTENSIONS)
 
    if ignored_images:
        print(f"WARNING: {len(ignored_images)} file(s) in {images_dir} don't have a supported "
              f"extension {IMAGE_EXTENSIONS} and were ignored: {ignored_images[:5]}"
              + (" ..." if len(ignored_images) > 5 else ""))
 
    images, masks = [], []
    skipped_no_mask = []
    skipped_padding = []
    for stem in sorted(image_files):
        image_filename = image_files[stem]
 
        if stem not in mask_files:
            skipped_no_mask.append(image_filename)   # no matching mask found -- see warning below
            continue
 
        image_path = os.path.join(images_dir, image_filename)
        mask_path = os.path.join(masks_dir, mask_files[stem])
 
        image_arr = load_image(image_path, target_size)
 
        if not is_mostly_valid(image_arr):
            skipped_padding.append(image_filename)   # too much black padding -- an edge-of-source-image tile
            continue
 
        images.append(image_arr)
        masks.append(load_mask(mask_path, target_size))
 
    if skipped_no_mask:
        print(f"WARNING: {len(skipped_no_mask)} image(s) had no matching mask and were skipped: "
              f"{skipped_no_mask[:5]}" + (" ..." if len(skipped_no_mask) > 5 else ""))
 
    if skipped_padding:
        print(f"WARNING: {len(skipped_padding)} image(s) were more than {MAX_BLACK_FRACTION:.0%} black "
              f"padding (edge-of-source-image tiles) and were skipped: {skipped_padding[:5]}"
              + (" ..." if len(skipped_padding) > 5 else ""))
 
    images = np.stack(images) if images else np.empty((0, target_size[1], target_size[0], 3))
    masks = np.stack(masks) if masks else np.empty((0, target_size[1], target_size[0], 1))
 
    print(f"Loaded {len(images)} image/mask pairs from {images_dir} "
          f"(shape: {images.shape})")
 
    return images, masks
 
 
# --------------------------------------------------------------------------
# 4. OPTIONAL: split one folder of images+masks into train/val/test
# --------------------------------------------------------------------------
def split_dataset(source_dir, output_dir, train_frac=0.7, val_frac=0.15, seed=42):
    """If you only have ONE folder of images/masks (no split yet), this copies
    files into train/val/test for you in the given proportions.
 
    source_dir must contain `images/` and `masks/` subfolders.
    output_dir will be created with the layout `load_dataset()` expects:
        output_dir/images/<split>/...
        output_dir/masks/<split>/...
    Only images that have a matching mask (paired by filename stem) are used.
    """
    import shutil
    import random
 
    images_dir = os.path.join(source_dir, "images")
    masks_dir = os.path.join(source_dir, "masks")
 
    image_files, _ = _index_by_stem(images_dir, IMAGE_EXTENSIONS)
    mask_files, _ = _index_by_stem(masks_dir, MASK_EXTENSIONS)
    stems = sorted(stem for stem in image_files if stem in mask_files)
 
    random.seed(seed)
    random.shuffle(stems)
 
    n_train = int(len(stems) * train_frac)
    n_val = int(len(stems) * val_frac)
 
    splits = {
        "train": stems[:n_train],
        "val": stems[n_train:n_train + n_val],
        "test": stems[n_train + n_val:],
    }
 
    for split_name, split_stems in splits.items():
        for kind in ("images", "masks"):
            os.makedirs(os.path.join(output_dir, kind, split_name), exist_ok=True)
 
        for stem in split_stems:
            shutil.copy(os.path.join(images_dir, image_files[stem]),
                        os.path.join(output_dir, "images", split_name, image_files[stem]))
            shutil.copy(os.path.join(masks_dir, mask_files[stem]),
                        os.path.join(output_dir, "masks", split_name, mask_files[stem]))
 
        print(f"{split_name}: {len(split_stems)} image/mask pairs")
 
 
# --------------------------------------------------------------------------
# 5. QUICK CHECK
# --------------------------------------------------------------------------
if __name__ == "__main__":
    # TODO: replace with your actual Kaggle input path -- run the os.walk
    # snippet shown in chat to find it, e.g.:
    # /kaggle/input/hutan-royal-belum-dataset/Dataset
    DATASET_ROOT = "/kaggle/input/datasets/orcazyx/hutan-royal-belum-dataset-3-0/Hutan-Royal-Belum-Dataset-5.0/Dataset"
 
    train_images, train_masks = load_dataset(DATASET_ROOT, "train")
    val_images, val_masks = load_dataset(DATASET_ROOT, "val")
    test_images, test_masks = load_dataset(DATASET_ROOT, "test")
 
    print("Train:", train_images.shape, train_masks.shape)
    print("Val:  ", val_images.shape, val_masks.shape)
    print("Test: ", test_images.shape, test_masks.shape)

"""
Data Augmentation Generator — Forest / Non-Forest Segmentation
==================================================================
 
A faithful port of the `adjustData()` / `trainGenerator()` functions from
the original notebook (itself borrowed there from
github.com/bragagnololu/UNet-defmapping), adapted to your own dataset.
 
WHAT THIS DOES:
Rather than feeding the same fixed set of training images into the model
every epoch, this wraps Keras's `ImageDataGenerator` to apply random
augmentations (rotation, shifts, shear, zoom, flips) to each image AND
its matching mask *identically* every time a batch is drawn -- so the
model sees a slightly different version of your dataset each epoch. With
a small dataset, this is one of the cheapest ways to reduce overfitting.

"""


In [ ]:

import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# --------------------------------------------------------------------------
# 1. AUGMENTATION SETTINGS (same values as the original notebook)
# --------------------------------------------------------------------------
data_gen_args = dict(
    rotation_range=180,
    width_shift_range=0.25,
    height_shift_range=0.25,
    shear_range=0.25,
    zoom_range=0.25,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode="reflect",   # how to fill pixels revealed by a shift/rotation
)


# --------------------------------------------------------------------------
# 2. MASK CLEANUP AFTER AUGMENTATION
# --------------------------------------------------------------------------
def adjustData(img, mask, num_class=2):
    """Re-binarize the mask after augmentation may have blurred its edges."""
    mask[mask > 0.5] = 1   # FOREST
    mask[mask <= 0.5] = 0  # NON-FOREST
    return (img, mask)

# --------------------------------------------------------------------------
# 3. THE GENERATOR ITSELF
# --------------------------------------------------------------------------
def trainGenerator(
    batch_size,
    image_array,
    mask_array,
    aug_dict,
    image_save_prefix="image",
    mask_save_prefix="mask",
    num_class=2,
    save_to_dir=None,
    target_size=(224, 224),
    seed=1,
):

    image_datagen = ImageDataGenerator(**aug_dict)
    mask_datagen = ImageDataGenerator(**aug_dict)

    image_generator = image_datagen.flow(
        image_array,
        batch_size=batch_size,
        save_to_dir=save_to_dir,
        save_prefix=image_save_prefix,
        seed=seed,
    )

    mask_generator = mask_datagen.flow(
        mask_array,
        batch_size=batch_size,
        save_to_dir=save_to_dir,
        save_prefix=mask_save_prefix,
        seed=seed,   # <-- same seed as image_generator is what keeps them in sync
    )

    train_generator = zip(image_generator, mask_generator)

    for (img, mask) in train_generator:
        img, mask = adjustData(img, mask, num_class)
        yield (img, mask)


# --------------------------------------------------------------------------
# 4. QUICK CHECK
# --------------------------------------------------------------------------
if __name__ == "__main__":

    DATASET_ROOT = "/kaggle/input/datasets/orcazyx/hutan-royal-belum-dataset-3-0/Hutan-Royal-Belum-Dataset-5.0/Dataset"


    train_images, train_masks = load_dataset(DATASET_ROOT, "train")

    train_gen = trainGenerator(
        batch_size=4,
        image_array=train_images,
        mask_array=train_masks,
        aug_dict=data_gen_args,
    )

    # Pull exactly one batch out, just to confirm shapes/values look right
    sample_images, sample_masks = next(train_gen)
    print("Batch of images:", sample_images.shape, sample_images.dtype)
    print("Batch of masks: ", sample_masks.shape, sample_masks.dtype,
          "unique values:", np.unique(sample_masks))

"""
Train — Attention U-Net for Forest / Non-Forest Segmentation
================================================================
 
Wires together the three pieces built so far:
  - unet_models.py       (build_attention_unet)
  - data_loader.py        (load_dataset)
  - data_augmentation.py  (trainGenerator, data_gen_args)
 
...and runs model.fit(), saving the best-performing weights and the
training history to disk as it goes.
"""
 

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback
# --------------------------------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------------------------------
DATASET_ROOT = "/kaggle/input/datasets/orcazyx/hutan-royal-belum-dataset-3-0/Hutan-Royal-Belum-Dataset-5.0/Dataset"
OUTPUT_DIR = "/kaggle/working/models"   
 
BATCH_SIZE = 4
EPOCHS = 50
 

QUICK_TEST = False
QUICK_TEST_TRAIN_IMAGES = 200
QUICK_TEST_VAL_IMAGES = 40
QUICK_TEST_EPOCHS = 2
if QUICK_TEST:
    OUTPUT_DIR = OUTPUT_DIR + "_quicktest"
 

MODEL_NAMES_TO_TRAIN = ["abl_none", "abl_L1", "abl_L2","abl_L3","abl_L4","abl_all"]
 
unknown_models = [name for name in MODEL_NAMES_TO_TRAIN if name not in MODEL_REGISTRY]
if unknown_models:
    raise ValueError(f"Unknown model name(s) {unknown_models}. Available: {list(MODEL_REGISTRY)}")
MODELS_TO_TRAIN = {name: MODEL_REGISTRY[name] for name in MODEL_NAMES_TO_TRAIN}
 
# Every model starts from the same random seed, so differences between the
# placement variants come from where the gates are, not from a lucky start.
SEED = 42
 
# The five scores tracked every epoch (F1 is derived below, the rest come
# straight from the metrics compiled into each model in unet_models.py).
METRIC_NAMES = ["accuracy", "precision", "recall", "f1_score", "iou"]
 
 
# --------------------------------------------------------------------------
# PER-EPOCH SCORE HELPERS
# --------------------------------------------------------------------------
def add_f1_scores(hist):
    """Adds "f1_score" and "val_f1_score" lists to a Keras history dict.
 
    Precision and recall are pooled over the whole epoch, so
    F1 = 2PR / (P + R) gives exactly the pooled F1 (2TP / (2TP + FP + FN)).
    An epoch where the model predicts no forest at all has P = R = 0, which
    is reported as F1 = 0 rather than a divide-by-zero.
    """
    for prefix in ("", "val_"):
        p = np.array(hist[prefix + "precision"], dtype="float64")
        r = np.array(hist[prefix + "recall"], dtype="float64")
        denominator = p + r
        f1 = np.divide(2 * p * r, denominator, out=np.zeros_like(denominator), where=denominator > 0)
        hist[prefix + "f1_score"] = f1.tolist()
    return hist
 
 
def build_epoch_table(hist):
    """One row per epoch: loss, then the five scores on the training data,
    then the same five on the validation data."""
    table = pd.DataFrame({"epoch": np.arange(1, len(hist["loss"]) + 1)})
    table["loss"] = hist["loss"]
    table["val_loss"] = hist["val_loss"]
    for name in METRIC_NAMES:
        table["train_" + name] = hist[name]
    for name in METRIC_NAMES:
        table["val_" + name] = hist["val_" + name]
    return table
 
 
class EpochProgress(Callback):
    
    def __init__(self, model_name, total_epochs):
        super().__init__()
        self.model_name = model_name
        self.total_epochs = total_epochs
 
    def on_train_begin(self, logs=None):
        self.train_start = time.time()
        self.epoch_seconds = []
 
    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.time()
 
    def on_epoch_end(self, epoch, logs=None):
        self.epoch_seconds.append(time.time() - self.epoch_start)
        elapsed_hours = (time.time() - self.train_start) / 3600
        epochs_left = self.total_epochs - (epoch + 1)
        remaining_hours = np.mean(self.epoch_seconds) * epochs_left / 3600
        print(f"[{self.model_name}] epoch {epoch + 1}/{self.total_epochs} took "
              f"{self.epoch_seconds[-1] / 60:.1f} min | elapsed {elapsed_hours:.2f} h | "
              f"worst-case remaining for this model ~{remaining_hours:.2f} h", flush=True)
 
 
# --------------------------------------------------------------------------
# 2. LOAD DATA (shared across both models -- only needs to happen once)
# --------------------------------------------------------------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
train_images, train_masks = load_dataset(DATASET_ROOT, "train")
val_images, val_masks = load_dataset(DATASET_ROOT, "val")
 
if QUICK_TEST:
    train_images, train_masks = train_images[:QUICK_TEST_TRAIN_IMAGES], train_masks[:QUICK_TEST_TRAIN_IMAGES]
    val_images, val_masks = val_images[:QUICK_TEST_VAL_IMAGES], val_masks[:QUICK_TEST_VAL_IMAGES]
    EPOCHS = QUICK_TEST_EPOCHS
    print(f"QUICK TEST: {len(train_images)} train / {len(val_images)} val images, {EPOCHS} epochs. "
          f"Outputs go to {OUTPUT_DIR}")
 
# What the run is actually using -- the first lines to check in the log.
gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs visible to TensorFlow: {gpus if gpus else 'NONE -- training will run on the CPU and be extremely slow'}")
print(f"Train images: {len(train_images)} | validation images: {len(val_images)} | "
      f"batch size: {BATCH_SIZE} | steps per epoch: {max(1, len(train_images) // BATCH_SIZE)} | "
      f"max epochs per model: {EPOCHS} | models: {list(MODELS_TO_TRAIN)}", flush=True)
 
 
# --------------------------------------------------------------------------
# 3. TRAIN EACH MODEL IN TURN
# --------------------------------------------------------------------------
# NOTE on save_weights_only=True: attention_block() uses a custom Python
# function inside a Lambda layer, which Keras's full-model save format
# can't reliably reconstruct in a fresh session. Saving just the weights
# and rebuilding the architecture from unet_models.py each time sidesteps
# that entirely -- applied to both models here for consistency, even
# though only the attention model strictly needs it.
 
trained_histories = {}   # model_name -> history.history, used only if you want to compare later in this session
 
for model_name, build_fn in MODELS_TO_TRAIN.items():
    print(f"\n{'=' * 60}\nTraining: {model_name}\n{'=' * 60}")
 
    train_gen = trainGenerator(
        batch_size=BATCH_SIZE,
        image_array=train_images,
        mask_array=train_masks,
        aug_dict=data_gen_args,
    )
 
    tf.keras.utils.set_random_seed(SEED)   # identical starting point for every model
    model = build_fn()
    model.summary()
 
    checkpoint_path = os.path.join(OUTPUT_DIR, f"{model_name}_best.weights.h5")

    checkpoint = ModelCheckpoint(
        checkpoint_path,
        monitor="val_iou",
        mode="max",
        save_best_only=True,   # only overwrites the file when val_iou improves
        save_weights_only=True,
        verbose=1,
    )
 
    # Stops training once val_loss stops improving for `patience` epochs in a row,
    # and restores the best-performing weights seen during training (not just
    # whatever epoch happened to be last) -- guards against wasting epochs/GPU
    # time on a model that's already overfitting.
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1,
    )
 
    # Halves the learning rate when val_loss plateaus for `patience` epochs,
    # letting training take smaller, more precise steps once progress stalls
    # rather than continuing to overshoot with the original learning rate.
    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    )
 
    steps_per_epoch = max(1, len(train_images) // BATCH_SIZE)
 
    history = model.fit(
        train_gen,
        steps_per_epoch=steps_per_epoch,
        validation_data=(val_images, val_masks),
        epochs=EPOCHS,
        callbacks=[checkpoint, early_stopping, reduce_lr, EpochProgress(model_name, EPOCHS)],
        verbose=2,   # one line per epoch -- the default progress bar turns into a wall of text in a Save Version log
    )
 
    history_dict = add_f1_scores(history.history)
    trained_histories[model_name] = history_dict
 
    # Save the raw per-epoch history so you can reload and re-plot it later
    # without retraining
    np.save(os.path.join(OUTPUT_DIR, f"{model_name}_history.npy"), history_dict)
 
    # ----------------------------------------------------------------
    # Per-epoch score sheet (accuracy / precision / recall / F1 / IoU)
    # ----------------------------------------------------------------
    epoch_table = build_epoch_table(history_dict)
    epoch_csv_path = os.path.join(OUTPUT_DIR, f"{model_name}_epoch_scores.csv")
    epoch_table.to_csv(epoch_csv_path, index=False)
 
    print(f"\nPer-epoch scores for {model_name}:")
    print(epoch_table.round(4).to_string(index=False))
 
    best_row = epoch_table.loc[epoch_table["val_iou"].idxmax()]
    print(f"\nBest epoch by val_iou: {int(best_row['epoch'])} "
          f"(val IoU {best_row['val_iou']:.4f}, val F1 {best_row['val_f1_score']:.4f}, "
          f"val precision {best_row['val_precision']:.4f}, val recall {best_row['val_recall']:.4f})")
    print(f"Per-epoch scores saved to: {epoch_csv_path}")
 
    # ----------------------------------------------------------------
    # Plot this model's training curves: loss + the five scores
    # ----------------------------------------------------------------
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    panels = [("loss", "Loss")] + [(name, name.replace("_", " ").title()) for name in METRIC_NAMES]
    for ax, (key, title) in zip(axes.flatten(), panels):
        ax.plot(epoch_table["epoch"], history_dict[key], label="train")
        ax.plot(epoch_table["epoch"], history_dict["val_" + key], label="validation")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend()
    fig.suptitle(f"{model_name}: training vs validation, per epoch")
    plt.tight_layout()
    plt.show()
 
    print(f"Best {model_name} weights saved to: {checkpoint_path}")
 
 
print("\nAll models trained. Saved weights:")
for model_name in MODELS_TO_TRAIN:
    print(f"  {model_name}: {os.path.join(OUTPUT_DIR, model_name + '_best.weights.h5')}")

"""
Predict — Score the Trained Model Across the Whole Test Set
================================================================
 
Loads the model saved by train.py, runs it on every image in the test
set, and reports accuracy / precision / recall / F1 -- both per-image
(so you can spot which specific tiles the model struggles with) and
pooled across the entire test set (a single overall score).
 
KAGGLE NOTE: paste this into its own cell after data_loader.py has
already run in this session (load_dataset() needs to already be in
memory -- no local imports, same pattern as the earlier scripts).
"""
 

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, jaccard_score
 
 
# --------------------------------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------------------------------
DATASET_ROOT = "/kaggle/input/datasets/orcazyx/hutan-royal-belum-dataset-3-0/Hutan-Royal-Belum-Dataset-5.0/Dataset"
OUTPUT_DIR = "/kaggle/working/models"   # same folder train.py saves the weights to

PREDICTION_THRESHOLD = 0.5   # model output > this -> predicted "forest" (1)

# The five score columns that get averaged.
METRIC_COLUMNS = ["accuracy", "precision", "recall", "f1_score", "iou"]
 
# Models to score, by name (see MODEL_REGISTRY in unet_models.py). The names
# must match the ones used in train.py, since they decide each weights file's
# name. A model with no saved weights is skipped with a message, so this list
# can name every model even while only some have finished training.
MODEL_NAMES_TO_EVALUATE = ["abl_none", "abl_L1", "abl_L2", "abl_L3", "abl_L4", "abl_all"]
# Other names you can add: "unet", "attention_unet"

unknown_models = [name for name in MODEL_NAMES_TO_EVALUATE if name not in MODEL_REGISTRY]
if unknown_models:
    raise ValueError(f"Unknown model name(s) {unknown_models}. Available: {list(MODEL_REGISTRY)}")
MODELS_TO_EVALUATE = {name: MODEL_REGISTRY[name] for name in MODEL_NAMES_TO_EVALUATE}

# Folders searched, in order, for each model's <name>_best.weights.h5. If you
# trained the models in SEPARATE Save Version runs, add each run's output to
# this notebook as an input and list its folder here, e.g.
#   "/kaggle/input/<that-notebook's-output>/models"
# (run os.walk("/kaggle/input") to see the exact path).
WEIGHTS_DIRS = [OUTPUT_DIR]


def find_weights(model_name):
    """Path of the saved weights for `model_name`, or None if there are none."""
    for folder in WEIGHTS_DIRS:
        path = os.path.join(folder, f"{model_name}_best.weights.h5")
        if os.path.exists(path):
            return path
    return None
 
 
# --------------------------------------------------------------------------
# 2. LOAD TEST DATA (shared across both models -- only needs to happen once)
# --------------------------------------------------------------------------
test_images, test_masks = load_dataset(DATASET_ROOT, "test")
print(f"Loaded {len(test_images)} test images.")
 
 
# --------------------------------------------------------------------------
# 3. EVALUATE EACH MODEL IN TURN
# --------------------------------------------------------------------------
overall_results = {}   # model_name -> dict of overall (pooled) metrics, used for the comparison table
average_results = {}   # "model - tile group" -> dict of per-image average metrics, used for the average comparison table
 
for model_name, build_fn in MODELS_TO_EVALUATE.items():
    weights_path = find_weights(model_name)
    if weights_path is None:
        print(f"\nSKIPPED {model_name}: no {model_name}_best.weights.h5 found in {WEIGHTS_DIRS}")
        continue

    print(f"\n{'=' * 60}\nEvaluating: {model_name}\n{'=' * 60}")
 
    model = build_fn()
    model.load_weights(weights_path)
 
    raw_predictions = model.predict(test_images)                       # shape (N, H, W, 1), values 0-1
    predicted_masks = (raw_predictions > PREDICTION_THRESHOLD).astype("uint8")
 
    # ---- Per-image metrics ----
    per_image_rows = []
    for i in range(len(test_images)):
        true_flat = test_masks[i].flatten().astype("uint8")
        pred_flat = predicted_masks[i].flatten()
 
        per_image_rows.append({
            "image_index": i,
            "accuracy": accuracy_score(true_flat, pred_flat),
            "precision": precision_score(true_flat, pred_flat, zero_division=0),
            "recall": recall_score(true_flat, pred_flat, zero_division=0),
            "f1_score": f1_score(true_flat, pred_flat, zero_division=0),
            "iou": jaccard_score(true_flat, pred_flat, zero_division=0),
            "true_forest_fraction": float(true_flat.mean()),   # how much of this tile is really forest
        })
 
    per_image_df = pd.DataFrame(per_image_rows)
 
    # ---- Average of every score across the test images ----
    # A tile whose true mask has NO forest scores 0 for precision, recall, F1
    # and IoU even when the model predicts it perfectly (there is no forest to
    # find, so those scores are undefined and zero_division=0 turns them into
    # 0). So the average is reported two ways:
    #   - all tiles:          every test image counts equally
    #   - tiles with forest:  only images that really contain forest, which is
    #                         the fairer picture of how well forest is detected
    has_forest = per_image_df["true_forest_fraction"] > 0
    average_all = per_image_df[METRIC_COLUMNS].mean()
    average_forest = per_image_df.loc[has_forest, METRIC_COLUMNS].mean()

    average_results[f"{model_name} - all tiles"] = average_all.to_dict()
    average_results[f"{model_name} - tiles with forest"] = average_forest.to_dict()

    # ---- Overall metrics (every pixel from every test image pooled together) ----
    all_true = test_masks.flatten().astype("uint8")
    all_pred = predicted_masks.flatten()
 
    overall_metrics = {
        "accuracy": accuracy_score(all_true, all_pred),
        "precision": precision_score(all_true, all_pred, zero_division=0),
        "recall": recall_score(all_true, all_pred, zero_division=0),
        "f1_score": f1_score(all_true, all_pred, zero_division=0),
        "iou": jaccard_score(all_true, all_pred, zero_division=0),
    }
    overall_results[model_name] = overall_metrics
 
    # ---- Report this model ----
    print("\nPer-image summary (mean / std / min / max across all test images):")
    print(per_image_df[METRIC_COLUMNS].describe().loc[["mean", "std", "min", "max"]])

    print(f"\nAverage per image ({len(per_image_df)} test images, {int(has_forest.sum())} of them contain forest):")
    print(pd.DataFrame({
        "all tiles": average_all,
        "tiles with forest": average_forest,
    }).T.to_string(float_format=lambda value: f"{value:.4f}"))
 
    print("\nOverall (pooled across every pixel in the test set):")
    for metric_name, value in overall_metrics.items():
        print(f"  {metric_name:10s}: {value:.4f}")
 
    # Save this model's sheet: one row per image, then the two average rows at the bottom
    average_rows = pd.DataFrame([
        {"image_index": "average (all tiles)",
         **average_all.to_dict(),
         "true_forest_fraction": per_image_df["true_forest_fraction"].mean()},
        {"image_index": "average (tiles with forest)",
         **average_forest.to_dict(),
         "true_forest_fraction": per_image_df.loc[has_forest, "true_forest_fraction"].mean()},
    ])
    sheet_df = pd.concat([per_image_df, average_rows], ignore_index=True)

    csv_path = os.path.join(OUTPUT_DIR, f"{model_name}_test_set_scores.csv")
    sheet_df.to_csv(csv_path, index=False)
    print(f"\nPer-image scores (with average rows) saved to: {csv_path}")
 
 
# --------------------------------------------------------------------------
# 4. SIDE-BY-SIDE COMPARISON ACROSS MODELS
# --------------------------------------------------------------------------
if not overall_results:
    raise RuntimeError(f"No trained weights found for any of {MODEL_NAMES_TO_EVALUATE} in {WEIGHTS_DIRS}.")

comparison_df = pd.DataFrame(overall_results).T   # rows = model name, columns = metric
comparison_df.index.name = "model"
 
print(f"\n{'=' * 60}\nModel comparison (overall, pooled metrics)\n{'=' * 60}")
print(comparison_df)
 
comparison_path = os.path.join(OUTPUT_DIR, "model_comparison.csv")
comparison_df.to_csv(comparison_path)
print(f"\nComparison table saved to: {comparison_path}")

average_comparison_df = pd.DataFrame(average_results).T[METRIC_COLUMNS]   # rows = "model - tile group"
average_comparison_df.index.name = "model - tiles averaged"

print(f"\n{'=' * 60}\nModel comparison (average of per-image scores)\n{'=' * 60}")
print(average_comparison_df)

average_comparison_path = os.path.join(OUTPUT_DIR, "model_comparison_average.csv")
average_comparison_df.to_csv(average_comparison_path)
print(f"\nAverage comparison table saved to: {average_comparison_path}")

"""
Prediction Result Analysis — Visual Comparisons
===================================================
 
For a handful of test images per model, plots four things side by side:
  1. The original image
  2. The ground-truth mask
  3. The model's predicted mask
  4. An error map, color-coding where the prediction was right or wrong
 
"""

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
 
 
# --------------------------------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------------------------------
DATASET_ROOT = "/kaggle/input/datasets/orcazyx/hutan-royal-belum-dataset-3-0/Hutan-Royal-Belum-Dataset-5.0/Dataset"
OUTPUT_DIR = "/kaggle/working/models"
 
PREDICTION_THRESHOLD = 0.5
NUM_SAMPLES = 5       # how many test images to visualize per model
RANDOM_SEED = 42      # fixed seed so the same sample images get picked every run -- makes results comparable across models
 
# The test set measured at ~10% forest overall, and picking purely random tiles
# turned up several that were ~0% forest -- technically correct, but not useful
# to look at (nothing for the error map to actually show). This filters sample
# selection to tiles that contain at least some forest, so comparisons are
# actually informative. Lower this if too few tiles qualify for your dataset.
MIN_FOREST_FRACTION = 0.05
 
# Models to draw, by name (see MODEL_REGISTRY in unet_models.py). A model with
# no saved weights is skipped with a message.
MODEL_NAMES_TO_ANALYZE = ["abl_none", "abl_L1", "abl_L2", "abl_L3", "abl_L4", "abl_all"]
# Other names you can add: "unet", "attention_unet"
 
unknown_models = [name for name in MODEL_NAMES_TO_ANALYZE if name not in MODEL_REGISTRY]
if unknown_models:
    raise ValueError(f"Unknown model name(s) {unknown_models}. Available: {list(MODEL_REGISTRY)}")
MODELS_TO_ANALYZE = {name: MODEL_REGISTRY[name] for name in MODEL_NAMES_TO_ANALYZE}
 
# Folders searched, in order, for each model's <name>_best.weights.h5 (same
# idea as in predict.py -- add other runs' output folders here if the models
# were trained in separate Save Version runs).
WEIGHTS_DIRS = [OUTPUT_DIR]
 
 
def find_weights(model_name):
    """Path of the saved weights for `model_name`, or None if there are none."""
    for folder in WEIGHTS_DIRS:
        path = os.path.join(folder, f"{model_name}_best.weights.h5")
        if os.path.exists(path):
            return path
    return None
 
 
# --------------------------------------------------------------------------
# 2. LOAD TEST DATA + PICK SAMPLE IMAGES (shared across both models)
# --------------------------------------------------------------------------
test_images, test_masks = load_dataset(DATASET_ROOT, "test")
print(f"Loaded {len(test_images)} test images.")
 
rng = np.random.default_rng(RANDOM_SEED)
 
forest_fractions = test_masks.mean(axis=(1, 2, 3))   # fraction of forest pixels, per test image
candidate_indices = np.where(forest_fractions >= MIN_FOREST_FRACTION)[0]
 
if len(candidate_indices) == 0:
    print(f"WARNING: no test images have >= {MIN_FOREST_FRACTION:.0%} forest pixels -- "
          f"falling back to picking from the full test set (samples may show little/no forest).")
    candidate_indices = np.arange(len(test_images))
 
sample_count = min(NUM_SAMPLES, len(candidate_indices))
sample_indices = rng.choice(candidate_indices, size=sample_count, replace=False)
sample_indices.sort()   # just for a tidier, predictable print/plot order
 
 
# --------------------------------------------------------------------------
# 3. ERROR MAP BUILDER
# --------------------------------------------------------------------------
def build_error_map(true_mask, pred_mask):
    """Returns an (H, W, 3) RGB image color-coding TP/TN/FP/FN pixels."""
    true_flat = true_mask.squeeze().astype("uint8")
    pred_flat = pred_mask.squeeze().astype("uint8")
 
    height, width = true_flat.shape
    rgb = np.zeros((height, width, 3), dtype="uint8")
 
    true_positive = (true_flat == 1) & (pred_flat == 1)
    true_negative = (true_flat == 0) & (pred_flat == 0)
    false_positive = (true_flat == 0) & (pred_flat == 1)
    false_negative = (true_flat == 1) & (pred_flat == 0)
 
    rgb[true_positive] = [255, 255, 255]   # white
    rgb[true_negative] = [0, 0, 0]         # black
    rgb[false_positive] = [255, 0, 0]      # red
    rgb[false_negative] = [255, 255, 0]    # yellow
 
    return rgb
 
 
# --------------------------------------------------------------------------
# 4. VISUALIZE EACH MODEL
# --------------------------------------------------------------------------
for model_name, build_fn in MODELS_TO_ANALYZE.items():
    weights_path = find_weights(model_name)
    if weights_path is None:
        print(f"\nSKIPPED {model_name}: no {model_name}_best.weights.h5 found in {WEIGHTS_DIRS}")
        continue
 
    print(f"\n{'=' * 60}\nAnalyzing: {model_name}\n{'=' * 60}")
 
    model = build_fn()
    model.load_weights(weights_path)
 
    sample_images = test_images[sample_indices]
    sample_true_masks = test_masks[sample_indices]
 
    raw_predictions = model.predict(sample_images)
    sample_pred_masks = (raw_predictions > PREDICTION_THRESHOLD).astype("uint8")
 
    fig, axes = plt.subplots(sample_count, 4, figsize=(14, 3.2 * sample_count))
    if sample_count == 1:
        axes = axes.reshape(1, -1)   # keep indexing consistent when there's only one row
 
    column_titles = ["Image", "True Mask", "Predicted Mask", "Error Map"]
 
    for row, idx in enumerate(sample_indices):
        image = sample_images[row].squeeze()
        true_mask = sample_true_masks[row]
        pred_mask = sample_pred_masks[row]
        error_map = build_error_map(true_mask, pred_mask)
 
        axes[row, 0].imshow(image)   # color image -- no cmap needed (masks/error map below are still single-channel)
        axes[row, 1].imshow(true_mask.squeeze(), cmap="gray")
        axes[row, 2].imshow(pred_mask.squeeze(), cmap="gray")
        axes[row, 3].imshow(error_map)
 
        for col in range(4):
            axes[row, col].axis("off")
            if row == 0:
                axes[row, col].set_title(column_titles[col])
 
        axes[row, 0].text(-0.15, 0.5, f"test #{idx}", transform=axes[row, 0].transAxes,
                           rotation=90, va="center", ha="center", fontsize=9)
 
    fig.suptitle(
        f"{model_name} — sample predictions "
        f"(error map: white=correct forest, black=correct non-forest, "
        f"red=false forest, yellow=missed forest)",
        fontsize=11,
    )
    plt.tight_layout()
 
    save_path = os.path.join(OUTPUT_DIR, f"{model_name}_visual_comparison.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
 
    print(f"Saved visual comparison to: {save_path}")